# load packages

In [ ]:
import pandas as pd

# read in meta data

In [ ]:
eval_sample_sets = pd.read_csv('prs_variability/CKD/meta_data/pgs_all_metadata_evaluation_sample_sets.csv')
eval_sample_sets.head()

In [ ]:
traits = pd.read_csv('prs_variability/CKD/meta_data/pgs_all_metadata_efo_traits.csv')
traits.head()

In [ ]:
cohorts = pd.read_csv('prs_variability/CKD/meta_data/pgs_all_metadata_cohorts.csv')
cohorts.head()

In [ ]:
performance = pd.read_csv('prs_variability/CKD/meta_data/pgs_all_metadata_performance_metrics.csv')
performance.head()

In [ ]:
pubs = pd.read_csv('prs_variability/CKD/meta_data/pgs_all_metadata_publications.csv')
pubs.head()

In [ ]:
score_dev_samples = pd.read_csv('prs_variability/CKD/meta_data/pgs_all_metadata_score_development_samples.csv')
score_dev_samples.head()

In [ ]:
scores = pd.read_csv('prs_variability/CKD/meta_data/pgs_all_metadata_scores.csv')
scores.head()

# filter trait to CKD

In [ ]:
traits[traits['Ontology Trait ID'].isin(['EFO_0003884'])]

In [ ]:
traits[traits['Ontology Trait Label'].str.contains('kidney')]

# filter scores to CKD

In [ ]:
ckd_scores = scores[scores['Mapped Trait(s) (EFO ID)'].isin(['EFO_0003884'])]
print(len(ckd_scores.index))
ckd_scores.head()

In [ ]:
ckd_scores['Reported Trait'].unique()

# filter eval sample sets to CKD

In [ ]:
ckd_eval_sample_sets = eval_sample_sets[eval_sample_sets['Polygenic Score (PGS) ID'].isin(ckd_scores['Polygenic Score (PGS) ID'])]
print(len(ckd_eval_sample_sets.index))
print(ckd_eval_sample_sets['Cohort(s)'].unique())
ckd_eval_sample_sets.head()

In [ ]:
ckd_eval_cohorts = ckd_eval_sample_sets[['Cohort(s)']].drop_duplicates()
ckd_eval_cohorts.head()

In [ ]:
ckd_eval_cohorts = ckd_eval_cohorts.assign(UNIQUE_COHORTS=ckd_eval_cohorts['Cohort(s)'].str.split('|')).explode('UNIQUE_COHORTS').reset_index(drop=True)
ckd_eval_cohorts.head()

In [ ]:
ckd_eval_cohorts=ckd_eval_cohorts[['UNIQUE_COHORTS']].drop_duplicates().dropna()
ckd_eval_cohorts

In [ ]:
ckd_eval_cohort_names=cohorts[cohorts['Cohort ID'].isin(ckd_eval_cohorts['UNIQUE_COHORTS'])]
ckd_eval_cohort_names

# filter score training samples

In [ ]:
ckd_score_dev_samples = score_dev_samples[score_dev_samples['Polygenic Score (PGS) ID'].isin(ckd_scores['Polygenic Score (PGS) ID'])].sort_values(by=['Polygenic Score (PGS) ID'])
print(len(ckd_score_dev_samples.index))
ckd_score_dev_samples

In [ ]:
ckd_train_cohorts = ckd_score_dev_samples[['Cohort(s)']].drop_duplicates()
ckd_train_cohorts = ckd_train_cohorts.assign(UNIQUE_COHORTS=ckd_train_cohorts['Cohort(s)'].str.split('|')).explode('UNIQUE_COHORTS').reset_index(drop=True)
ckd_train_cohorts = ckd_train_cohorts[['UNIQUE_COHORTS']].drop_duplicates().dropna()
ckd_train_cohorts

In [ ]:
ckd_train_cohort_names=cohorts[cohorts['Cohort ID'].isin(ckd_train_cohorts['UNIQUE_COHORTS'])]
ckd_train_cohort_names

# filter performance to CKD

In [ ]:
performance[performance['Evaluated Score'].isin(ckd_scores['Polygenic Score (PGS) ID'])]

# identify scores with distinct or same publications

In [ ]:
len(ckd_scores['Polygenic Score (PGS) ID'].unique())

In [ ]:
len(ckd_scores['PGS Publication (PGP) ID'].unique())

In [ ]:
ckd_scores[['Polygenic Score (PGS) ID','PGS Publication (PGP) ID']].sort_values(by=['PGS Publication (PGP) ID'])

In [ ]:
dup_pub = ckd_scores[ckd_scores['PGS Publication (PGP) ID'].isin(['PGP000517'])]
dup_pub[['Polygenic Score (PGS) ID',
         'PGS Name',
         'PGS Development Method',
         'Number of Variants',
         'PGS Publication (PGP) ID',
         'Publication (PMID)',
         'Ancestry Distribution (%) - Score Development/Training']]

In [ ]:
ckd_dup_pub = score_dev_samples[score_dev_samples['Polygenic Score (PGS) ID'].isin(dup_pub['Polygenic Score (PGS) ID'])]
ckd_dup_pub[['Polygenic Score (PGS) ID',
             'Number of Individuals',
             'Broad Ancestry Category',
             'GWAS Catalog Study ID (GCST...)',
             'Source PubMed ID (PMID)',
             'Cohort(s)']]

In [ ]:
ckd_dup_pub_id = ckd_dup_pub[['Polygenic Score (PGS) ID']]
ckd_dup_pub_id.to_csv('prs_variability/CKD/meta_data/CKD.PGS_list.PublicationID.PGP000517.txt',
                      sep='\t',
                      header=None,
                      index=None)

In [ ]:
no_dup_pub = ckd_scores[~ckd_scores['Polygenic Score (PGS) ID'].isin(dup_pub['Polygenic Score (PGS) ID'])]
no_dup_pub[['Polygenic Score (PGS) ID',
            'PGS Name',
            'PGS Development Method',
            'Number of Variants',
            'PGS Publication (PGP) ID',
            'Publication (PMID)',
            'Ancestry Distribution (%) - Score Development/Training']]

In [ ]:
score_dev_samples.columns

In [ ]:
ckd_no_dup_pub = score_dev_samples[score_dev_samples['Polygenic Score (PGS) ID'].isin(no_dup_pub['Polygenic Score (PGS) ID'])]
ckd_no_dup_pub[['Polygenic Score (PGS) ID',
                'Number of Individuals',
                'Broad Ancestry Category',
                'GWAS Catalog Study ID (GCST...)',
                'Source PubMed ID (PMID)',
                'Cohort(s)']]

In [ ]:
ckd_ukbb = ckd_dup_pub_id.copy()
ckd_ukbb.loc[len(ckd_ukbb)] = ['PGS001272']
ckd_ukbb.loc[len(ckd_ukbb)] = ['PGS002237']
print(len(ckd_ukbb.index))
print(len(ckd_dup_pub_id.index))
ckd_ukbb.tail()

In [ ]:
ckdgen_eur = pd.DataFrame(data={'PGS' : ['PGS004158',
                                         'PGS004142',
                                         'PGS004128',
                                         'PGS004112',
                                         'PGS004101',
                                         'PGS004088',
                                         'PGS004058',
                                         'PGS004045',
                                         'PGS004030',
                                         'PGS004016',
                                         'PGS004004',
                                         'PGS003988',
                                         'PGS004074',
                                         'PGS000728',
                                         'PGS000859',
                                         'PGS002757']})
ckdgen_eur.to_csv('prs_variability/CKD/meta_data/CKD.PGS_list.Training.CKDGen.EUR_only.txt',
                  header=None,
                  index=None)
print(len(ckdgen_eur.index))
ckdgen_eur.head()

In [ ]:
ckdgen_multi = pd.DataFrame(data={'PGS' : ['PGS002237',
                                          'PGS004889',
                                          'PGS005113']})
ckdgen_multi.to_csv('prs_variability/CKD/meta_data/CKD.PGS_list.Training.CKDGen.multi_ancestry.txt',
                  header=None,
                  index=None)
print(len(ckdgen_multi.index))
ckdgen_multi.head()

In [ ]:
ukbb = pd.DataFrame(data={'PGS' : ['PGS001272',
                                  'PGS004224']})
ukbb.to_csv('prs_variability/CKD/meta_data/CKD.PGS_list.Training.UKBB.txt',
                  header=None,
                  index=None)
print(len(ukbb.index))
ukbb.head()

In [ ]:
mvp_regards = pd.DataFrame(data={'PGS' : ['PGS005090']})
mvp_regards.to_csv('prs_variability/CKD/meta_data/CKD.PGS_list.Training.MVP_REGARDS_meta.txt',
                  header=None,
                  index=None)
print(len(mvp_regards.index))
mvp_regards.head()